# Python Accounting Automation
This notebook demonstrates a Python-based reconciliation workflow for comparing ERP transaction exports against accounting shadow-book records. Ideally, this helps ensure perfect accuracy at all times.  The workflow uses `pandas` to clean accounting data, standardize account numbers, compare balances, and flag exceptions for review. This project uses sanitized sample data and is intended as a portfolio demonstration of accounting automation, financial analysis, and data cleaning. 

## Save Two Files in CSV format
  <b> Simple! This is the ONLY thing we need do to do.</b>  Python will do the rest! Python can read these excel sheets on the backend. They must be saved in the correct Property/Location spot. To find the location of an Excel file on your computer do the following:
1. Right click on Excel sheet
2. Copy paste the Location
3. End with the excel file title name.
    
In the end, it should look like the below: <br>
    `access=r'C:\Users\scott\OneDrive\Desktop\access.csv'`
    
 <br> Note: Due to CSV files saving long numbers in scientific notation, it is recommended to save the file as a .xlsx file.

In [ ]:
access=r'C:\Users\sschm\Desktop\automate\access.xlsx'
data=r'C:\Users\sschm\Desktop\automate\data.xlsx'

In [ ]:
#NOTHING TO BE CHANGED BELOW THIS!
import csv, time
import pandas as pd
from datetime import date

#DISPLAY REPORT ON TODAY'S DATE:
print("Running Data Analysis report on: ", date.today().strftime("%m-%d-%Y") )
start=time.time()

## View Original Data
Using .head() can show the columns and first five rows of a csv file.

In [ ]:
dataDF=pd.read_excel(data)
dataDF.head()

## View Original Access Data
Using .head() can show the columns and first five rows of a csv file.

In [ ]:
accessDF=pd.read_excel(access) 
accessDF.head()

# Group Transaction Data by Accounting Subtotals
The pandas library will read the excel csv files. The things that happen are as follows:
1. Create an ending Amount column by subtracting debits and credits.
2. Filter the GL Account and Ending Amount. These are the only two columns we need.
3. Use a dataframe groupby for each accounting object code to get all the accounting subtotals.
4. Temporarily delete hypens because Access does not have any hypens.

#### a. Create an ending Amount column by subtracting debits and credits. 
Most YTD Credits will be NA which is equilavent to 0. 

In [ ]:
dataDF['YTD Credits'].fillna(0, inplace=True)
dataDF['Ending Amount'] = dataDF.apply(lambda x: x['YTD Debits'] - x['YTD Credits'], axis = 1)
dataDF.head()

#### b. Filter the GL Account and Ending Amount. These are the two columns Python needs.

In [ ]:
dataDF=dataDF.filter(['GL Account', 'Ending Amount'])
dataDF.head()

#### c. Use a dataframe groupby for each accounting object code to get all the accounting subtotals.

In [ ]:
#dataList=dataDF.to_dict(orient='list')
dataTotals=dataDF.groupby('GL Account')['Ending Amount'].sum().reset_index()
dataTotals.head()

#### d. Temporarily delete hypens because Access does not have any hypens. Python and Excel will not recognize account numbers as a match unless they are exactly the same. 

In [ ]:
dataTotals['GL Account'] = dataTotals['GL Account'].str.replace('-', '') 
dataTotals.head()

# Group Access Data by Accounting Subtotals
This is a sample shadowbook. nornally in the form of SQL or Access. Here is a sample of what most Access shadow book data could look like. The same procedures will be used as the data subtotals. Now look at the accounting subtotals on data and access. We can clearly see that the first three subtotals balance correctly. The last account nunmber does not balance.

In [ ]:
accessDF=accessDF.filter(['ACCOUNT ID', 'INVOICE AMOUNT', 'REQ#']) #Only need these 3 columns
accessTotals = accessDF.groupby('ACCOUNT ID')['INVOICE AMOUNT'].sum().reset_index() #Sum totals by account number
accessTotals['ACCOUNT ID'] = accessTotals['ACCOUNT ID'].str.replace('-', '') #gets rid of hyphens 
accessTotals.head()

# Compare Access and Real Data 
Do the real transaction accounting object codes and shadowbook accounting subtoals match? Python will soon find out! Note: To all the expert computer scientists out there, I do realize I could of use a join statement. However, in the original draft I did it the probably more complex way using two for loops. A join statement would likely make the code more readable.

In [ ]:
for d in dataTotals['GL Account']: #checks each data account 
    for a in accessTotals['ACCOUNT ID']: # checks each account in Access...
        if d==a: #if Data and Access Account are the same then do the following:
            dRow=dataTotals.loc[dataTotals['GL Account']==d]
            aRow=accessTotals.loc[accessTotals['ACCOUNT ID']==a]
            dSUM=round(float(dRow['Ending Amount'].values), 2) #data Subtotal Number
            aSUM=round(float(aRow['INVOICE AMOUNT'].values),2) #access Subtotal Number
            if  dSUM==aSUM: 
                continue; #nothing else needs to be done.
                #hyphen=a[0:5]+"-"+a[5:7]+"-"+a[7:16]+"-"+a[16:] #adds hyphens to account number, easier to copy/paste
                #print(d, dSUM, "data matches access: ", aSUM, hyphen)
            else:
                hyphens=a[0:5]+"-"+a[5:7]+"-"+a[7:16]+"-"+a[16:] #adds hyphens to account number, easier to copy/paste
                print(a, " Access-----check account----- Data", hyphens)
                #print(dSUM, " data DOES NOT match access ", aSUM, ". Difference is: ", round(float(dSUM-aSUM), 2) 
print("Finished matching data to access accounts in: ", round(time.time()-start, 3), " seconds");
print("----------END SECTION1 ----------")

#### Python correctly noticed that the account number 55555-55-555555555-555555 does not balance. 

# Check for Missing D in Req Column
Has something posted? When something has posted as a real data transaction, Access 'Req' column must be updated with a D. Normally, this takes a great deal of time by having to check EVERY individual account. When a new transaction with real data is recorded, a D is required in REQ# column in Access. Case must equal true, otherwise dec will be accidently pulled onto the dataFrame when it should not be.

In [ ]:
accessDF=accessDF.dropna(subset=['REQ#'])
accessDF=accessDF.loc[accessDF['REQ#'].str.contains("D", case=True)] #keeps only access rows with a "D"
accessDF.head()

In [ ]:
accessDsum=accessDF.groupby('ACCOUNT ID')['INVOICE AMOUNT'].sum().reset_index()
accessDsum['ACCOUNT ID'] = accessDsum['ACCOUNT ID'].str.replace('-', '') #gets rid of hyphens 
accessDsum.head()

#### Compare Data to Access Posted Accounts

In [ ]:
#GET EVERY REAL DATA ACCOUNTING NUMBER USED:
dataAccounts=dataDF['GL Account'].unique() #finds only unique accounts
dataAcc=[] #final list

for acc in dataAccounts:
    dataAcc.append(acc.replace('-', '') ) #removes hypens

#CHECK DATA FOR UNBANACED ACCOUNTS:
for account in accessDsum['ACCOUNT ID']: # checks each account in Access...
    aRow=accessDsum.loc[accessDsum['ACCOUNT ID']==account]
    dRow=dataTotals.loc[dataTotals['GL Account']==d]
    accessDtotal=round(float(aRow['INVOICE AMOUNT'].values),2) #access Subtotal Number
    dataTotal=round(float(dRow['Ending Amount'].values), 2) #datat Subtotal Number
    if account in dataAcc: #only need to check for Data Accounts 
        hyphens=account[0:5]+"-"+account[5:7]+"-"+account[7:16]+"-"+account[16:] #see note at end
        if accessDtotal==dataTotal:
            continue; #nothing else is needed to be done.
            #print(account, " is likely correct ", hyphens)
        elif accessDtotal<dataTotal:
            print(account, " Missing Ds in Access in REQ# column: ", hyphens)
        elif accessDtotal>dataTotal:
            print(account, " account likely has too many D in REQ column: ", hyphens)
        else:
            print(account, "Error likely happened on account: ", hyphens) #Note: This should never happen
print("------Completed Section 2--------")
print("Done. Finished comparing Access with data in ", round(time.time()-start, 3), " seconds")

## Special Notes
Line 80 Special Note: The only reason I add back the hyphens is so an accounting assistant can easily copy/paste the account number into a database to retrieve the records. Otherwise, the individual would have to type out the full account number by hand and/or could possibly accidently type the wrong account number. This not only saves time but also makes the work of the accountant more enjoyable. 

## Diclosure
ACCOUNTING AUTOMATION SHOULD NEVER REPLACE AN ACCOUNTANT, CPA OR FINANCIAL ADVISOR: <br>
Python can work at instant speeds to detect new transactions, human error, transpoisitional errors, or missing transactions. It may not detect every error, fraud, or have the mindset of a human. However, Python can help with the following:
1. Compare thousands of transactions between actual processed transactions and accounting books. Are the accounting subtotals balanced?
2. New transactions that have been posted or changed. Has a new transaction been posted? Has an old transaction not been recorded?
3. Finding a transpositional error. Was a transaction recorded as 3832.12 but was actually 3823.12? This is difficult to see with the human eye when there are thousands of transactions.
4. Calculating all possible subtotals. How did our potentially ending balance end at XX.XX?